In [25]:
from datasets import load_dataset
import pandas as pd

In [26]:
!wget https://math-qa.github.io/math-QA/data/MathQA.zip
!unzip -o MathQA.zip

--2026-06-12 06:07:31--  https://math-qa.github.io/math-QA/data/MathQA.zip
Resolving math-qa.github.io (math-qa.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to math-qa.github.io (math-qa.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7302821 (7.0M) [application/x-zip-compressed]
Saving to: ‘MathQA.zip.1’

MathQA.zip.1        100%[===================>]   6.96M  26.4MB/s    in 0.3s    

2026-06-12 06:07:31 (26.4 MB/s) - ‘MathQA.zip.1’ saved [7302821/7302821]

Archive:  MathQA.zip
  inflating: constant_list.txt       
  inflating: __MACOSX/._constant_list.txt  
  inflating: operation_list.txt      
  inflating: __MACOSX/._operation_list.txt  
  inflating: test.json               
  inflating: __MACOSX/._test.json    
  inflating: dev.json                
  inflating: __MACOSX/._dev.json     
  inflating: train.json              
  inflating: __MACOSX/._train.json   
  inflating: challenge_test.json   

In [27]:
import json

In [28]:
with open("train.json", "r") as f:
    train_data = json.load(f)

In [29]:
train_df = pd.DataFrame(train_data)

In [30]:
print(train_df.shape)
train_df.head()

(29837, 7)


,Problem,Rationale,options,correct,annotated_formula,linear_formula,category
0,the banker ' s gain of a certain sum due 3 yea...,"""explanation : t = 3 years r = 10 % td = ( bg ...","a ) rs . 400 , b ) rs . 300 , c ) rs . 500 , d...",a,"divide(multiply(const_100, divide(multiply(36,...","multiply(n2,const_100)|multiply(n0,n1)|divide(...",gain
1,average age of students of an adult school is ...,"""explanation : let the original no . of studen...","a ) 1200 , b ) 120 , c ) 360 , d ) 240 , e ) n...",d,"multiply(divide(subtract(multiply(add(32, 4), ...","add(n2,n3)|multiply(n1,n2)|multiply(n1,#0)|sub...",general
2,sophia finished 2 / 3 of a book . she calculat...,let xx be the total number of pages in the boo...,"a ) 229 , b ) 270 , c ) 877 , d ) 266 , e ) 281",b,"divide(90, subtract(const_1, divide(2, 3)))","divide(n0,n1)|subtract(const_1,#0)|divide(n2,#1)",general
3,120 is what percent of 50 ?,"""50 * x = 120 - - > x = 2.4 - - > 2.4 expresse...","a ) 5 % , b ) 240 % , c ) 50 % , d ) 2 % , e )...",b,"multiply(divide(120, 50), const_100)","divide(n0,n1)|multiply(#0,const_100)|",gain
4,there are 10 girls and 20 boys in a classroom ...,"if girls is 10 and boys is 20 , then 10 / 20 ....","a ) 1 / 2 , b ) 1 / 3 , c ) 1 / 5 , d ) 10 / 3...",a,"divide(10, 20)","divide(n0,n1)",other


In [31]:
train_df.describe()

,Problem,Rationale,options,correct,annotated_formula,linear_formula,category
count,29837,29837,29837,29837,29837,29837,29837
unique,29837,29537,25515,5,26250,10721,6
top,7.51 8.22 7.86 8.36 8.09 7.83 8.30 8.01 7.73 8...,answer : a,"a ) 1 , b ) 2 , c ) 3 , d ) 4 , e ) 5",c,"add(const_3, const_4)","divide(n0,n1)|multiply(#0,const_100)|",general
freq,1,12,143,6653,26,184,13273


In [32]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29837 entries, 0 to 29836
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Problem            29837 non-null  object
 1   Rationale          29837 non-null  object
 2   options            29837 non-null  object
 3   correct            29837 non-null  object
 4   annotated_formula  29837 non-null  object
 5   linear_formula     29837 non-null  object
 6   category           29837 non-null  object
dtypes: object(7)
memory usage: 1.6+ MB


In [33]:
train_df.isnull().sum()

,0
Problem,0
Rationale,0
options,0
correct,0
annotated_formula,0
linear_formula,0
category,0


In [34]:
print(train_df.columns)

Index(['Problem', 'Rationale', 'options', 'correct', 'annotated_formula',
       'linear_formula', 'category'],
      dtype='object')


In [35]:
train_df.duplicated().sum()

np.int64(0)

In [36]:
def parse_mathqa_formula(formula):
    """
    Converts allenai/math_qa linear_formula into an executable Python script block.
    Example Input: "multiply(n2,const_100)|multiply(n0,n1)|divide(#0,#1)|"
    """
    if not isinstance(formula, str) or not formula:
        return ""

    # Split the operations by the pipe character
    operations = [op for op in formula.strip().split('|') if op]
    python_lines = ["```python"]

    for i, op_str in enumerate(operations):
        try:
            # Extract the operation name and the arguments inside the parentheses
            op_name = op_str.split('(')[0]
            args_str = op_str.split('(')[1].replace(')', '')
            args = args_str.split(',')
        except IndexError:
            continue

        parsed_args = []
        for arg in args:
        arg = arg.strip()
        if arg.startswith('const_'):
            # Convert 'const_100' to '100', or 'const_0_5' to '0.5'
            val = arg.replace('const_', '').replace('_', '.')
            parsed_args.append(val)
        elif arg.startswith('#'):
            # Convert '#0' to 'step_0' (referencing a previous calculation)
            step_idx = arg.replace('#', '')
            parsed_args.append(f"step_{step_idx}")
        else:
            # Keep variables like n0, n1, n2 as they are
            parsed_args.append(arg)

        # Map the abstract operations to Python syntax
        if len(parsed_args) == 2:
            if op_name == 'add':
                line = f"step_{i} = {parsed_args[0]} + {parsed_args[1]}"
            elif op_name == 'subtract':
                line = f"step_{i} = {parsed_args[0]} - {parsed_args[1]}"
            elif op_name == 'multiply':
                line = f"step_{i} = {parsed_args[0]} * {parsed_args[1]}"
            elif op_name == 'divide':
                line = f"step_{i} = {parsed_args[0]} / {parsed_args[1]}"
            elif op_name == 'power':
                line = f"step_{i} = {parsed_args[0]} ** {parsed_args[1]}"
            else:
                line = f"step_{i} = {op_name}({parsed_args[0]}, {parsed_args[1]})"
        else:
            # Fallback for unexpected formats
            line = f"step_{i} = {op_name}({', '.join(parsed_args)})"

        python_lines.append(line)

    # The final answer is always the result of the very last step
    last_step = len(operations) - 1
    python_lines.append(f"result = step_{last_step}")
    python_lines.append("```")

    return "\n".join(python_lines)

IndentationError: expected an indented block after 'for' statement on line 23 (1837284838.py, line 24)

In [ ]:
clean_mqa = pd.DataFrame()

clean_mqa['id'] = "mqa_" + train_df.index.astype(str).str.zfill(5)
clean_mqa['dataset_source'] = "math_qa"
clean_mqa['split'] = "train"

clean_mqa['question'] = train_df['Problem']

In [ ]:
clean_mqa['code'] = train_df['linear_formula'].apply(parse_mathqa_formula)

In [ ]:
clean_mqa['answer'] = ""

In [ ]:
clean_mqa['question'] = clean_mqa['question'].astype(str).str.strip()

clean_mqa = clean_mqa[
    (clean_mqa['question'] != '') &
    (clean_mqa['code'] != '')
]

In [ ]:
initial_rows = len(clean_mqa)

clean_mqa = clean_mqa.drop_duplicates(
    subset=['question'],
    keep='first'
)

print(
    f"Removed {initial_rows - len(clean_mqa)} duplicate questions"
)

In [ ]:
clean_mqa = clean_mqa.dropna(
    subset=['question', 'code']
)

In [ ]:
clean_mqa.shape

In [ ]:
clean_mqa = clean_mqa.head(4000)

print("Final rows:", len(clean_mqa))

In [ ]:
expected_columns = [
    'id',
    'dataset_source',
    'split',
    'question',
    'code',
    'answer'
]

clean_mqa = clean_mqa[expected_columns]

In [ ]:
output_filename = "unified_mqa_train.csv"

clean_mqa.to_csv(
    output_filename,
    index=False
)

print(
    f"✅ Saved {len(clean_mqa)} rows to {output_filename}"
)

In [ ]:
display(clean_mqa.head())

clean_mqa.info()

In [ ]:
# from google.colab import files

# files.download('unified_mqa_train.csv')